# Módulo 11.1 — Autoencoders
**Sergio Gevatschnaider** · laboratorio sintético, reproducible.

Objetivo: comparar PCA con un autoencoder neuronal 2→1→2 entrenado realmente, evaluar reconstrucción fuera de muestra y explicar el error de anomalías. Los datos no corresponden a empresas reales.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch
from torch import nn
np.random.seed(21); torch.manual_seed(21)
n=650
x=np.random.normal(0,1,n)
X=np.c_[x,0.7*x+0.28*np.sin(3*x)+np.random.normal(0,.16,n)]
train_idx, test_idx=train_test_split(np.arange(n),test_size=.25,random_state=21)
scaler=StandardScaler().fit(X[train_idx])
Xtr=scaler.transform(X[train_idx]).astype('float32')
Xte=scaler.transform(X[test_idx]).astype('float32')
print('train',Xtr.shape,'test',Xte.shape)

## Línea base: PCA
Se ajusta exclusivamente en el conjunto de entrenamiento. No comparar pérdidas in-sample contra test sin advertencia.

In [ ]:
pca=PCA(n_components=1).fit(Xtr)
Xhat_pca=pca.inverse_transform(pca.transform(Xte))
mse_pca=np.mean((Xte-Xhat_pca)**2)
print('PCA MSE test:',round(mse_pca,5))

## Autoencoder no lineal: entrenamiento y validación

In [ ]:
model=nn.Sequential(nn.Linear(2,6),nn.Tanh(),nn.Linear(6,1),nn.Tanh(),nn.Linear(1,6),nn.Tanh(),nn.Linear(6,2))
opt=torch.optim.Adam(model.parameters(),lr=0.01,weight_decay=1e-4)
tr=torch.tensor(Xtr); te=torch.tensor(Xte)
history=[]
for epoch in range(180):
    model.train(); opt.zero_grad()
    loss=((model(tr)-tr)**2).mean(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad(): test_loss=((model(te)-te)**2).mean().item()
    history.append((loss.item(),test_loss))
with torch.no_grad(): recon=model(te).numpy()
mse_ae=np.mean((Xte-recon)**2)
print('PCA test MSE:',round(mse_pca,5),'AE test MSE:',round(mse_ae,5))
plt.plot(history);plt.legend(['train','test']);plt.title('Curva de aprendizaje (MSE)');plt.xlabel('Época');plt.show()

## Reconstrucción y anomalías
El error de reconstrucción no demuestra fraude: define un puntaje que requiere calibración, supervisión y análisis de falsos positivos.

In [ ]:
fig,ax=plt.subplots(figsize=(7,5));ax.scatter(Xte[:,0],Xte[:,1],s=10,label='original');ax.scatter(recon[:,0],recon[:,1],s=10,alpha=.5,label='reconstrucción');ax.legend();plt.show()
score=np.mean((Xte-recon)**2,axis=1)
cut=np.quantile(np.mean((Xtr-model(tr).detach().numpy())**2,axis=1),.95)
print('Umbral exploratorio sobre train:',round(cut,5),'alertas en test:',int((score>cut).sum()))

## Reflexión
1. ¿Es la mejora del AE sobre PCA robusta a semillas y arquitectura?
2. ¿Qué resulta del uso de datos no vistos?
3. ¿Cómo calibrarías el costo de alertas?
4. ¿En qué casos el código latente no es interpretable?